In [1]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.parsing.metadata_utils import *
from maomao.parsing.integrated_dataset_utils import *

#### Toxic dataset integration and label-consistency analysis
- This notebook builds an integrated, sequence-level toxicity dataset by merging peptide annotations from a large collection of preprocessed sources exported in PATH_EXPORT (e.g., BIOPEP-UWM, CAPTP, CICERON, CSM-Toxin, HyPepTox-Fuse, iAMPCN, MultiPep, Pep-Lab_db, PeptiTox, PLPTP, ProToxin, SATPdb, StrucToxNet, tAMPer, ToxDL 2.0, ToxGIN, ToxIBTL, ToxinPred (1/2/3), ToxiPep, ToxMSRC, ToxTeller, TPpred-LE, UniDL4BioPep, Zhao et al., and Peptipedia2.0), plus an additional “unlabeled” set from ProToxin. The inputs are CSV files containing at minimum a peptide sequence and a source-specific toxicity label column (standardized to toxic).

- All unique peptide sequences across sources are collected and used to construct a pivot table, where each row is a unique sequence and each column corresponds to one source. Source labels are mapped onto this pivot table using a consistent encoding scheme (positive/negative/unlabeled/unknown), with missing annotations filled as 999 to explicitly track absence of evidence per source.

- Before label integration, the notebook performs sequence-level quality control. It removes peptides containing non-canonical amino acids and filters sequences outside global length bounds (MIN_LENGTH_SEQUENCE, MAX_LENGTH_SEQUENCE). The notebook records the number of sequences retained after each filter and summarizes the final length distribution (min/max/mean/median) for reporting and reproducibility.

- After building the pivot table, cross-source label consistency is quantified per sequence. The notebook counts the number of positive, negative, unlabeled, and unknown annotations, computes positive/negative vote percentages considering only labeled sources, and derives high-level flags identifying sequences that are exclusively positive, exclusively negative, unlabeled-only, or ambiguous (conflicting evidence). Ambiguous sequences are further stratified into bins based on their positive-vote percentage to provide a graded view of disagreement severity.

- The main outputs are exported to ../../dataset_post_processing/toxic/ as non-overlapping CSV subsets (negative.csv, only_negative.csv, positive.csv, only_positive.csv, ambiguous_data.csv) and a metadata.json file summarizing the input sources, filtering statistics, and label-consistency metrics. These outputs are designed to support downstream model training, benchmarking, and audit of label agreement across heterogeneous toxicity resources.

In [2]:
name_task = "toxic_effect_classification"
output_folder = "../../processed_data/integrating_and_cleaning_data/toxic"

# PATH_EXPORT are imported from peptide_toxicity_classifier.constants.
# Update them in constants.py according to the required input and export paths.

- Reading all sources

In [3]:
df_BIOPEP_UWM_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/BIOPEP-UWM/processed_toxic_dataset.csv")
df_BIOPEP_UWM_toxic = df_BIOPEP_UWM_toxic.rename(columns={"label": "toxic"})

In [4]:
df_CAPTP = pd.read_csv(f"{PATH_EXPORT}/{name_task}/CAPTP/processed_toxic_dataset.csv")
df_CAPTP = df_CAPTP.rename(columns={"label": "toxic"})

In [5]:
df_CICERON_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/CICERON/processed_toxic_dataset.csv")
df_CICERON_toxic = df_CICERON_toxic.rename(columns={"label": "toxic"})

In [6]:
df_csm_toxin_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/csm-toxin/processed_toxic_dataset.csv")
df_csm_toxin_toxic = df_csm_toxin_toxic.rename(columns={"label": "toxic"})

In [7]:
df_HyPepTox_Fuse_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/HyPepTox-Fuse/processed_toxic_dataset.csv")
df_HyPepTox_Fuse_toxic = df_HyPepTox_Fuse_toxic.rename(columns={"label": "toxic"})

In [8]:
df_iAMPCN_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/iAMPCN/processed_toxic_dataset.csv")
df_iAMPCN_toxic = df_iAMPCN_toxic.rename(columns={"label": "toxic"})

In [9]:
df_MultiPep_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/MultiPep/processed_toxic_dataset.csv")
df_MultiPep_toxic = df_MultiPep_toxic.rename(columns={"label": "toxic"})

In [10]:
df_Pep_Lab_db_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Pep-Lab_db/processed_toxic_dataset.csv")
df_Pep_Lab_db_toxic = df_Pep_Lab_db_toxic.rename(columns={"label": "toxic"})

In [11]:
df_peptidereactor_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/peptidereactor/processed_toxic_dataset.csv")
df_peptidereactor_toxic = df_peptidereactor_toxic.rename(columns={"label": "toxic"})

In [12]:
df_PeptiTox_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/PeptiTox/processed_toxic_dataset.csv")
df_PeptiTox_toxic = df_PeptiTox_toxic.rename(columns={"label": "toxic"})

In [13]:
df_PLPTP_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/PLPTP/processed_toxic_dataset.csv")
df_PLPTP_toxic = df_PLPTP_toxic.rename(columns={"label": "toxic"})

In [14]:
df_ProToxin_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ProToxin/processed_toxic_dataset.csv")
df_ProToxin_toxic = df_ProToxin_toxic.rename(columns={"label": "toxic"})

In [15]:
df_SATPdb_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/SATPdb/processed_toxic_dataset.csv")
df_SATPdb_toxic = df_SATPdb_toxic.rename(columns={"label": "toxic"})

In [16]:
df_StrucToxNet_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/StrucToxNet/processed_toxic_dataset.csv")
df_StrucToxNet_toxic = df_StrucToxNet_toxic.rename(columns={"label": "toxic"})

In [17]:
df_tAMPer_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/tAMPer/processed_toxic_dataset.csv")
df_tAMPer_toxic = df_tAMPer_toxic.rename(columns={"label": "toxic"})

In [18]:
df_ToxDL2_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ToxDL 2.0/processed_toxic_dataset.csv")
df_ToxDL2_toxic = df_ToxDL2_toxic.rename(columns={"label": "toxic"})

In [19]:
df_ToxGIN_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ToxGIN/processed_toxic_dataset.csv")
df_ToxGIN_toxic = df_ToxGIN_toxic.rename(columns={"label": "toxic"})

In [20]:
df_ToxIBTL_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ToxIBTL/processed_toxic_dataset.csv")
df_ToxIBTL_toxic = df_ToxIBTL_toxic.rename(columns={"label": "toxic"})

In [21]:
df_ToxinPred_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ToxinPred/processed_toxic_dataset.csv")
df_ToxinPred_toxic = df_ToxinPred_toxic.rename(columns={"label": "toxic"})

In [22]:
df_ToxinPred2_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ToxinPred 2.0/processed_toxic_dataset.csv")
df_ToxinPred2_toxic = df_ToxinPred2_toxic.rename(columns={"label": "toxic"})

In [23]:
df_ToxinPred3_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ToxinPred 3.0/processed_toxic_dataset.csv")
df_ToxinPred3_toxic = df_ToxinPred3_toxic.rename(columns={"label": "toxic"})

In [24]:
df_ToxiPep_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ToxiPep/processed_toxic_dataset.csv")
df_ToxiPep_toxic = df_ToxiPep_toxic.rename(columns={"label": "toxic"})

In [25]:
df_ToxMSRC_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ToxMSRC/processed_toxic_dataset.csv")
df_ToxMSRC_toxic = df_ToxMSRC_toxic.rename(columns={"label": "toxic"})

In [26]:
df_ToxTeller_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ToxTeller/processed_toxic_dataset.csv")
df_ToxTeller_toxic = df_ToxTeller_toxic.rename(columns={"label": "toxic"})

In [27]:
df_TPpred_LE_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/TPpred-LE/processed_toxic_dataset.csv")
df_TPpred_LE_toxic = df_TPpred_LE_toxic.rename(columns={"label": "toxic"})

In [28]:
df_UniDL4BioPep_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/UniDL4BioPep/processed_toxic_dataset.csv")
df_UniDL4BioPep_toxic = df_UniDL4BioPep_toxic.rename(columns={"label": "toxic"})

In [29]:
df_Zhao_et_al_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Zhao et al./processed_toxic_dataset.csv")
df_Zhao_et_al_toxic = df_Zhao_et_al_toxic.rename(columns={"label": "toxic"})

In [30]:
df_peptipedia_toxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Peptipedia2.0/processed_neurotoxic_dataset.csv")
df_peptipedia_toxic = df_peptipedia_toxic.rename(columns={"label": "toxic"})

- Reading all sources with unlabel

In [31]:
df_protoxin_unlabel = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ProToxin/detected_unlabel_sequences.csv")
df_protoxin_unlabel = df_protoxin_unlabel.rename(columns={"label": "toxic"})

- Collecting all sequences for activity

In [32]:
df_list_toxic = [
    df_BIOPEP_UWM_toxic, df_CAPTP, df_CICERON_toxic,
    df_csm_toxin_toxic, df_HyPepTox_Fuse_toxic, df_iAMPCN_toxic,
    df_MultiPep_toxic, df_Pep_Lab_db_toxic, df_peptidereactor_toxic, 
    df_PeptiTox_toxic, df_PLPTP_toxic, df_ProToxin_toxic,
    df_SATPdb_toxic, df_StrucToxNet_toxic, df_tAMPer_toxic,
     df_ToxDL2_toxic, df_ToxGIN_toxic, df_ToxIBTL_toxic,
    df_ToxinPred_toxic, df_ToxinPred2_toxic, df_ToxinPred3_toxic,
    df_ToxiPep_toxic, df_ToxMSRC_toxic, df_ToxTeller_toxic, 
    df_TPpred_LE_toxic, df_UniDL4BioPep_toxic, df_Zhao_et_al_toxic,
    df_peptipedia_toxic, df_protoxin_unlabel
]
unique_sequence_toxic = count_unique_sequence(df_list_toxic)

302451


- Create pivote dataset

In [33]:
df_pivote = create_pivote(unique_sequence_toxic)

- Removing sequences with non canonical residues 

In [34]:
n_before_canon = df_pivote.shape[0]
df_pivote["is_canon"] = df_pivote["sequence"].apply(check_sequence)
n_after_canon = df_pivote[df_pivote["is_canon"]].shape[0]

In [35]:
print(df_pivote["is_canon"].value_counts())
df_pivote = df_pivote[df_pivote["is_canon"]]

is_canon
True     302016
False       435
Name: count, dtype: int64


- Filter sequences by length

In [36]:
df_pivote["length"] = df_pivote["sequence"].str.len()
df_pivote["length"].describe()

count    302016.000000
mean        338.007003
std         391.971481
min           2.000000
25%         115.000000
50%         258.000000
75%         438.000000
max       35213.000000
Name: length, dtype: float64

In [37]:
n_before_length = n_after_canon
df_pivote["filter_length"] = df_pivote["length"].apply(check_length)
n_after_length = df_pivote[df_pivote["filter_length"]].shape[0]

In [38]:
df_pivote["filter_length"].value_counts()

filter_length
False    248288
True      53728
Name: count, dtype: int64

In [39]:
length_series = df_pivote[df_pivote["filter_length"]]["length"]

length_dist = {
    "min": length_series.min(),
    "max": length_series.max(),
    "mean": length_series.mean(),
    "median": length_series.median()
}

In [40]:
df_pivote = df_pivote[df_pivote["filter_length"]]
df_pivote.shape

(53728, 4)

In [41]:
df_pivote = df_pivote.drop(columns=["is_canon", "filter_length", "length"])

In [42]:
df_list_toxic = [
        ("BIOPEP-UWM", df_BIOPEP_UWM_toxic),
        ("CAPTP", df_CAPTP),
        ("CICERON", df_CICERON_toxic),
        ("CSM-Toxin", df_csm_toxin_toxic),
        ("HyPepTox-Fuse", df_HyPepTox_Fuse_toxic),
        ("iAMPCN", df_iAMPCN_toxic),
        ("MultiPep", df_MultiPep_toxic),
        ("Pep-Lab_db", df_Pep_Lab_db_toxic),
        ("peptidereactor", df_peptidereactor_toxic),
        ("PeptiTox", df_PeptiTox_toxic),
        ("PLPTP", df_PLPTP_toxic),
        ("ProToxin", df_ProToxin_toxic),
        ("SATPdb", df_SATPdb_toxic),
        ("StrucToxNet", df_StrucToxNet_toxic),
        ("tAMPer", df_tAMPer_toxic),
        ("ToxDL 2.0", df_ToxDL2_toxic),
        ("ToxGIN", df_ToxGIN_toxic),
        ("ToxIBTL", df_ToxIBTL_toxic),
        ("ToxinPred", df_ToxinPred_toxic),
        ("ToxinPred 2.0", df_ToxinPred2_toxic),
        ("ToxinPred 3.0", df_ToxinPred3_toxic),
        ("ToxiPep", df_ToxiPep_toxic),
        ("ToxMSRC", df_ToxMSRC_toxic),
        ("ToxTeller", df_ToxTeller_toxic),
        ("TPpred-LE", df_TPpred_LE_toxic),
        ("UniDL4BioPep", df_UniDL4BioPep_toxic),
        ("Zhao et al.", df_Zhao_et_al_toxic),
        ("Peptipedia2.0", df_peptipedia_toxic),
        ("ProToxin_unlabel", df_protoxin_unlabel)

]

In [43]:
for source, dataset in df_list_toxic:
    dataset = dataset[["sequence", "toxic"]]
    dataset = dataset.drop_duplicates(subset="sequence")
    mapping = dataset.set_index("sequence")["toxic"]

    # Mapear sin explotar memoria
    df_pivote[source] = (
        df_pivote["sequence"]
        .map(mapping)
        .fillna(999)
        .astype("int16")
    )

In [44]:
df_pivote.head(5)

,sequence,BIOPEP-UWM,CAPTP,CICERON,CSM-Toxin,HyPepTox-Fuse,iAMPCN,MultiPep,Pep-Lab_db,peptidereactor,...,ToxinPred 2.0,ToxinPred 3.0,ToxiPep,ToxMSRC,ToxTeller,TPpred-LE,UniDL4BioPep,Zhao et al.,Peptipedia2.0,ProToxin_unlabel
10,GWVYHAHPEANSFWT,999,999,999,999,1,999,999,999,999,...,999,1,999,999,999,999,999,999,999,999
16,MDIITLGWVGVLSVFTLSIAFVVWGRHGM,999,0,999,999,0,999,999,999,999,...,999,0,0,1,999,999,999,999,999,999
24,MSLLPVMVIFGLSFPPVFFELLVPLALFFLLRRLLQPTGIYDFVWH...,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,999,999,999,999,999
28,INWKKWWQVFYTVV,999,999,999,999,999,0,999,999,999,...,999,999,999,999,999,999,999,999,999,999
29,MIPVRCLSCGKPVSAYFNEYQRRVADGEDPKDVLDDLGLKRYCCRR...,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,999,999


- Working with pivote for detecting ambiguous sequences 

In [45]:
df_pivote = process_count_labels(df_pivote)

In [46]:
df_pivote["negative"].value_counts() # Includes sources labeled as nevative (0) and unlabeled (2)

negative
True     37519
False    16209
Name: count, dtype: int64

In [47]:
df_pivote["exclusive_0"].value_counts()

exclusive_0
True     37504
False    16224
Name: count, dtype: int64

In [48]:
df_pivote["positive"].value_counts() # Includes sources labeled as positive (1) and unlabeled (2)

positive
False    45293
True      8435
Name: count, dtype: int64

In [49]:
df_pivote["exclusive_1"].value_counts()

exclusive_1
False    45383
True      8345
Name: count, dtype: int64

In [50]:
df_pivote["only_unlabel"].value_counts() # Includes only sources unlabeled (2)

only_unlabel
False    53723
True         5
Name: count, dtype: int64

In [51]:
df_pivote.sort_values(by="percentage_1", ascending=False)

,sequence,BIOPEP-UWM,CAPTP,CICERON,CSM-Toxin,HyPepTox-Fuse,iAMPCN,MultiPep,Pep-Lab_db,peptidereactor,...,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
83503,KIDGYPVDNWNCKRICWYNNKYCYDLCKGLKADSGYCWGWTLSCYC...,999,999,999,1,999,999,1,999,999,...,0,0,23,True,False,True,False,False,0.0,100.0
83505,FLSHIAGFLSNLF,999,999,999,999,1,999,999,999,999,...,0,0,26,True,False,True,False,False,0.0,100.0
83559,DSAAMHTEYDVIATDNCIPCSHPACGINRGKC,999,999,999,999,1,999,1,999,999,...,0,0,20,True,False,True,False,False,0.0,100.0
243650,QFCCGHYDCDFIPNVC,999,999,999,999,1,999,999,999,999,...,0,0,27,True,False,True,False,False,0.0,100.0
243621,KECKPDGEQCGITDHNDCCNSCVCPGGPYMRPWDMMLQRCKCGPKE,999,1,999,999,999,999,999,999,999,...,0,0,19,True,False,True,False,False,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118439,RACAAFALLQFTIPGGRHAVHHAGL,999,999,999,999,999,999,999,999,999,...,1,0,28,False,True,False,True,False,100.0,0.0
118440,MSMSYSLLIHKTRKWGHKFFKFIFKSYIGSTMKSLINKVKRYSFYE...,999,999,999,0,999,999,999,999,999,...,3,0,26,False,True,False,True,False,100.0,0.0
118443,LWGHIWNFVHGLI,999,999,999,999,999,0,999,999,999,...,1,0,28,False,True,False,True,False,100.0,0.0
118445,MEALVYTFLLVSTLGIIFFAIFFREPPKVPTKKGK,999,999,999,999,999,999,999,999,999,...,1,0,28,False,True,False,True,False,100.0,0.0


- Splitting data into only negative, only positive, and with amiguous data

In [52]:
negative = df_pivote[df_pivote["negative"]]

In [53]:
only_negative = df_pivote[df_pivote["exclusive_0"]]

In [54]:
positive = df_pivote[df_pivote["positive"]]

In [55]:
only_positive = df_pivote[df_pivote["exclusive_1"]]

In [56]:
only_unlabel = df_pivote[df_pivote["only_unlabel"]]

In [57]:
df_ambiguous = df_pivote[(df_pivote["positive"] == False) & (df_pivote["negative"] == False) & (df_pivote["only_unlabel"] == False)]

- Processing ambiguous data

In [58]:
df_ambiguous = categorize_percentage(df_ambiguous)

In [59]:
df_ambiguous["Category_pbb"].value_counts()

Category_pbb
10-20    2769
>0       1791
70-80     701
40-50     641
20-30     562
60-70     401
80-90     346
50-60     333
30-40     164
>90        61
Name: count, dtype: int64

- Working with metada

In [60]:
seq_stats = {
    "canonical": {
        "before": n_before_canon,
        "after": n_after_canon
    },
    "length": {
        "before": n_before_length,
        "after": n_after_length,
        "min": MIN_LENGTH_SEQUENCE,
        "max": MAX_LENGTH_SEQUENCE
    },
    "length_dist": length_dist
}

metadata = build_dataset_metadata(
    task="toxic",
    source_list=df_list_toxic,
    pivote_df=df_pivote,
    outputs={
        "only_positive": only_positive,
        "only_negative": only_negative,
        "ambiguous": df_ambiguous
    },
    seq_stats=seq_stats,
    filters={
        "canonical_residues": True,
        "length_filter": True
    }
)
metadata

{'task': 'toxic',
 'generated_at': '2026-09-04T20:37:02.882983',
 'sources': {'n_unique_sequences': {'BIOPEP-UWM': 13,
   'CAPTP': 7513,
   'CICERON': 9,
   'CSM-Toxin': 221634,
   'HyPepTox-Fuse': 11036,
   'iAMPCN': 16542,
   'MultiPep': 5934,
   'Pep-Lab_db': 249,
   'peptidereactor': 1104,
   'PeptiTox': 3864,
   'PLPTP': 7513,
   'ProToxin': 251783,
   'SATPdb': 4130,
   'StrucToxNet': 9544,
   'tAMPer': 5708,
   'ToxDL 2.0': 11434,
   'ToxGIN': 4410,
   'ToxIBTL': 14092,
   'ToxinPred': 19391,
   'ToxinPred 2.0': 35242,
   'ToxinPred 3.0': 11036,
   'ToxiPep': 9889,
   'ToxMSRC': 7513,
   'ToxTeller': 4329,
   'TPpred-LE': 2345,
   'UniDL4BioPep': 3864,
   'Zhao et al.': 1998,
   'Peptipedia2.0': 577,
   'ProToxin_unlabel': 753}},
 'filters': {'canonical_residues': {'applied': True},
  'length_filter': {'applied': True, 'min_length': 5, 'max_length': 70}},
 'sequence_statistics': {'canonical_filter': {'before': 302451,
   'after': 302016},
  'length_filter': {'before': 302016, 'a

- Exporting data

In [61]:
os.makedirs(output_folder, exist_ok=True)

In [62]:
with open(f"{output_folder}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [63]:
negative.shape

(37519, 41)

In [64]:
only_negative.shape

(37504, 41)

In [65]:
positive.shape

(8435, 41)

In [66]:
only_positive.shape

(8345, 41)

In [67]:
only_unlabel.shape

(5, 41)

In [68]:
df_ambiguous.shape

(7769, 42)

In [69]:
negative.to_csv(f"{output_folder}/negative.csv", index=False)
only_negative.to_csv(f"{output_folder}/only_negative.csv", index=False)

In [70]:
positive.to_csv(f"{output_folder}/positive.csv", index=False)
only_positive.to_csv(f"{output_folder}/only_positive.csv", index=False)

In [71]:
df_ambiguous.to_csv(f"{output_folder}/ambiguous_data.csv", index=False)

In [72]:
df_ambiguous.to_csv(f"{output_folder}/ambiguous_data.csv", index=False)